# 01 - Prepare the data and find the charging gap

**Question:** In which Indian states is public EV charging falling behind EV adoption?

**Two official data tables** (see `docs/DATA_SOURCES.md`):
1. **EVs registered by state**, 1 Apr 2019 to 31 Mar 2024, with total vehicles sold (Ministry of Heavy Industries, from Vahan data)
2. **Public EV charging stations by state and year**, FY2020-21 to FY2025-26 (to 1 Mar 2026), installed by the three Oil Marketing Companies (Rajya Sabha reply, 27 Mar 2026)

**Our approach:** compare like with like. EVs registered up to March 2024 are compared with charging stations installed up to March 2024. Stations installed after that show whether the build-out later moved towards the states that needed it.

Run the cells from top to bottom with **Shift + Enter**.

| Step | What we do |
|---|---|
| 1 | Load the two tables |
| 2 | Check them against the official totals (catches typing and parsing errors) |
| 3 | Standardise state names and merge the tables |
| 4 | Calculate the gap metrics |
| 5 | Classify states into four groups |
| 6 | Look at the results and draw charts |
| 7 | Save the clean files for SQL and Tableau |

## Step 1 - Load the two tables

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ev = pd.read_csv("../data/raw/ev_registrations_by_state_fy2019_24.csv")
ch = pd.read_csv("../data/raw/ev_charging_stations_omc_by_state_mar2026.csv")

print("EV table:", ev.shape, "| Charging-station table:", ch.shape)
ev.head(3)

In [ ]:
ch.head(3)

## Step 2 - Check the data against the official totals
Both source documents publish a total row. If our numbers add up to the same totals, we know nothing was mistyped.
For charging stations there is a second check: installs minus decommissioned stations must equal the "total installed" column in every row.

In [ ]:
YEARS = ["fy20_21", "fy21_22", "fy22_23", "fy23_24", "fy24_25", "fy25_26_to_1mar26"]

checks = {
    "EVs registered (Apr 2019 - Mar 2024)":  (ev["total_ev"].sum(), 3639617),
    "Total vehicles sold (same period)":     (ev["total_vehicles_sold"].sum(), 107531040),
    "Charging stations installed (net)":     (ch["total_installed_net"].sum(), 27737),
    "Charging stations operational":         (ch["operational"].sum(), 22753),
    "Charging stations decommissioned":      (ch["decommissioned"].sum(), 21),
}
for name, (ours, official) in checks.items():
    status = "OK" if ours == official else "MISMATCH"
    print(f"{name:42s} ours = {ours:>12,}   official = {official:>12,}   {status}")

official_years = [314, 2344, 4658, 7649, 11246, 1547]
print()
print("Yearly installs match the official row:", ch[YEARS].sum().tolist() == official_years)

row_ok = (ch[YEARS].sum(axis=1) - ch["decommissioned"]) == ch["total_installed_net"]
print("Rows where installs - decommissioned = total:", int(row_ok.sum()), "of", len(ch))

## Step 3 - Standardise state names and merge
The two tables spell some states differently (`JHARKHAND` vs `Jharkand`, `PUDUCHERRY` vs `Pondicherry`, `LAKSHADWEEP` vs `Lakshwadeep`).
The charging table also lists **Leh** separately from **Ladakh**, so we add them together.

**Telangana** has charging stations but no EV registration figures in the official EV table (Telangana runs its own registration system), so it cannot be compared and is left out of the merged table.

In [ ]:
ev["state"] = ev["state_raw"].str.title().replace({
    "Ut Of Dnh And Dd": "Dadra & Nagar Haveli and Daman & Diu",
    "Andaman & Nicobar Island": "Andaman & Nicobar Islands",
})

ch["state"] = ch["state_raw"].replace({
    "Andaman & Nicobar": "Andaman & Nicobar Islands",
    "Jharkand": "Jharkhand",
    "Lakshwadeep": "Lakshadweep",
    "Pondicherry": "Puducherry",
    "UT of D&NH and D&D": "Dadra & Nagar Haveli and Daman & Diu",
    "Leh": "Ladakh",
})
number_cols = YEARS + ["decommissioned", "total_installed_net", "operational"]
ch = ch.groupby("state", as_index=False)[number_cols].sum()

print("In the charging table but not the EV table:", sorted(set(ch["state"]) - set(ev["state"])))
print("In the EV table but not the charging table:", sorted(set(ev["state"]) - set(ch["state"])))

d = ev.rename(columns={"total_ev": "ev_total", "total_vehicles_sold": "vehicles_total"}).merge(ch, on="state", how="inner")
print("States and UTs in the merged table:", len(d))

## Step 4 - Calculate the gap metrics

| Metric | Meaning |
|---|---|
| `ev_penetration_pct` | EVs as % of all vehicles registered (adoption) |
| `chargers_to_fy24` | Charging stations installed FY2020-21 to FY2023-24 |
| `evs_per_charger_fy24` | EVs per charging station (higher = more crowded) |
| `ev_share_pct` / `charger_share_fy24_pct` | The state's share of all EVs / of all stations |
| `share_gap_pp` | Station share minus EV share in percentage points (negative = under-served) |
| `new_chargers_after_fy24` | Stations installed from April 2024 to 1 Mar 2026 |
| `new_vs_ev_share_pp` | Share of new stations minus share of EVs (negative = the new build-out under-served it) |
| `operational_rate_pct` | % of installed stations that are operational |

In [ ]:
d["ev_penetration_pct"] = 100 * d["ev_total"] / d["vehicles_total"]

d["chargers_to_fy24"] = d[["fy20_21", "fy21_22", "fy22_23", "fy23_24"]].sum(axis=1)
d["new_chargers_after_fy24"] = d[["fy24_25", "fy25_26_to_1mar26"]].sum(axis=1)

d["evs_per_charger_fy24"] = np.where(d["chargers_to_fy24"] > 0, d["ev_total"] / d["chargers_to_fy24"], np.nan)
d["operational_rate_pct"] = np.where(d["total_installed_net"] > 0, 100 * d["operational"] / d["total_installed_net"], np.nan)

d["ev_share_pct"] = 100 * d["ev_total"] / d["ev_total"].sum()
d["charger_share_fy24_pct"] = 100 * d["chargers_to_fy24"] / d["chargers_to_fy24"].sum()
d["new_charger_share_pct"] = 100 * d["new_chargers_after_fy24"] / d["new_chargers_after_fy24"].sum()

d["share_gap_pp"] = d["charger_share_fy24_pct"] - d["ev_share_pct"]
d["new_vs_ev_share_pp"] = d["new_charger_share_pct"] - d["ev_share_pct"]

nat_penetration = 100 * d["ev_total"].sum() / d["vehicles_total"].sum()
nat_ratio = d["ev_total"].sum() / d["chargers_to_fy24"].sum()
print(f"National EV penetration (merged states): {nat_penetration:.2f}%")
print(f"National EVs per charging station (to Mar 2024): {nat_ratio:.0f}")

## Step 5 - Classify the states
To avoid noise from tiny states (a state with 27 EVs and 36 chargers tells us little), we only classify states with **at least 20,000 EVs** (the "analysis set").
Two simple tests against the national figures:
- **High adoption:** EV penetration at or above the national figure
- **Thinly served:** EVs per charging station at or above the national figure

States that are both high-adoption and thinly served are the **priority gap** states.

In [ ]:
d["in_analysis_set"] = d["ev_total"] >= 20000

high_adoption = d["ev_penetration_pct"] >= nat_penetration
thinly_served = d["evs_per_charger_fy24"] >= nat_ratio

d["quadrant"] = np.select(
    [~d["in_analysis_set"],
     high_adoption & thinly_served,
     high_adoption & ~thinly_served,
     ~high_adoption & thinly_served],
    ["Too few EVs to classify", "Priority gap", "High adoption, well served", "Low adoption, thinly served"],
    default="Low adoption, well served",
)

print("States in the analysis set:", int(d["in_analysis_set"].sum()))
d["quadrant"].value_counts()

## Step 6 - The results

In [ ]:
A = d[d["in_analysis_set"]].copy()
cols = ["state", "ev_total", "ev_penetration_pct", "chargers_to_fy24", "evs_per_charger_fy24", "share_gap_pp", "quadrant"]
A.sort_values("evs_per_charger_fy24", ascending=False)[cols].round(2).reset_index(drop=True)

In [ ]:
# Did the build-out after March 2024 move towards the states with the most EVs?
cols = ["state", "ev_share_pct", "new_charger_share_pct", "new_vs_ev_share_pp", "new_chargers_after_fy24"]
A.sort_values("new_vs_ev_share_pp")[cols].round(1).head(8).reset_index(drop=True)

In [ ]:
# Which states have a lot of stations that are not operational?
big = d[d["total_installed_net"] >= 50]
big.sort_values("operational_rate_pct")[["state", "total_installed_net", "operational", "operational_rate_pct"]].round(1).head(8).reset_index(drop=True)

In [ ]:
top12 = A.sort_values("evs_per_charger_fy24", ascending=False).head(12).iloc[::-1]
plt.figure(figsize=(8, 5))
plt.barh(top12["state"], top12["evs_per_charger_fy24"])
plt.axvline(nat_ratio, linestyle="--", color="grey")
plt.xlabel("EVs per charging station (Apr 2019 - Mar 2024)")
plt.title("States where charging stations are most stretched")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 6))
plt.scatter(A["ev_penetration_pct"], A["evs_per_charger_fy24"])
for _, r in A.iterrows():
    plt.annotate(r["state"], (r["ev_penetration_pct"], r["evs_per_charger_fy24"]), fontsize=7, xytext=(3, 3), textcoords="offset points")
plt.axvline(nat_penetration, linestyle="--", color="grey")
plt.axhline(nat_ratio, linestyle="--", color="grey")
plt.xlabel("EV penetration (% of vehicles registered)")
plt.ylabel("EVs per charging station")
plt.title("Adoption vs charging pressure (dashed lines = national figures)")
plt.tight_layout()
plt.show()

In [ ]:
yearly = ch[YEARS].sum()
labels = ["FY20-21", "FY21-22", "FY22-23", "FY23-24", "FY24-25", "FY25-26\n(to 1 Mar)"]
plt.figure(figsize=(8, 4))
plt.bar(labels, yearly.values)
plt.ylabel("Charging stations installed (OMCs)")
plt.title("New public charging stations per year")
plt.tight_layout()
plt.show()

## Step 7 - Save the clean files
- `ev_state_metrics.csv`: one row per state with all the metrics (for SQL and the Tableau map and scatter plot)
- `charger_installs_by_year.csv`: one row per state and year (for SQL and the Tableau trend chart)

In [ ]:
metrics_cols = [
    "state", "ev_total", "vehicles_total", "ev_penetration_pct",
    "chargers_to_fy24", "new_chargers_after_fy24", "total_installed_net", "operational", "operational_rate_pct",
    "evs_per_charger_fy24", "ev_share_pct", "charger_share_fy24_pct", "share_gap_pp",
    "new_charger_share_pct", "new_vs_ev_share_pp", "in_analysis_set", "quadrant",
]
metrics = d[metrics_cols].copy()
metrics["in_analysis_set"] = metrics["in_analysis_set"].astype(int)
metrics = metrics.round(3)
metrics.to_csv("../data/clean/ev_state_metrics.csv", index=False, encoding="utf-8")

year_labels = {
    "fy20_21": "FY2020-21", "fy21_22": "FY2021-22", "fy22_23": "FY2022-23",
    "fy23_24": "FY2023-24", "fy24_25": "FY2024-25", "fy25_26_to_1mar26": "FY2025-26 (to 1 Mar)",
}
installs = d.melt(id_vars="state", value_vars=YEARS, var_name="year_col", value_name="chargers_installed")
installs["fiscal_year"] = installs["year_col"].map(year_labels)
installs = installs[["state", "fiscal_year", "chargers_installed"]].sort_values(["state", "fiscal_year"])
installs.to_csv("../data/clean/charger_installs_by_year.csv", index=False, encoding="utf-8")

print("Saved ev_state_metrics.csv:", metrics.shape)
print("Saved charger_installs_by_year.csv:", installs.shape)

## Checkpoint
Send me the printed output of Steps 2, 3 and 5, plus the first table in Step 6 (EVs per charger by state).